# Project 01 — Data collection and first profiling

## First-Time Buyer Affordability Pressure by Area

This notebook collects the official ONS housing affordability workbook and performs the first profiling checks.

The main analytical measure for this project is the lower-quartile house price to lower-quartile workplace-based earnings ratio. This is the best starting point for a first-time buyer affordability angle because it focuses on the entry-level end of the market rather than only using median values.


## Analyst note

At this stage we are not trying to produce insights. We are making the data collection repeatable and checking whether the source is suitable for analysis.

A good interviewer would expect you to explain:
- where the data came from;
- whether the source is official and current;
- what the key measure means;
- what checks you ran before analysing it;
- what limitations you noticed.


In [ ]:
from pathlib import Path
from urllib.request import urlretrieve
from datetime import date
import re

import pandas as pd
import numpy as np


In [ ]:
PROJECT_ROOT = Path("..").resolve()
RAW_DIR = PROJECT_ROOT / "data" / "raw"
CLEANED_DIR = PROJECT_ROOT / "data" / "cleaned"

RAW_DIR.mkdir(parents=True, exist_ok=True)
CLEANED_DIR.mkdir(parents=True, exist_ok=True)

SOURCE_URL = (
    "https://www.ons.gov.uk/file?uri="
    "%2Fpeoplepopulationandcommunity%2Fhousing%2Fdatasets"
    "%2Fratioofhousepricetoworkplacebasedearningslowerquartileandmedian"
    "%2Fcurrent%2Faff1ratioofhousepricetoworkplacebasedearnings.xlsx"
)

RAW_FILE = RAW_DIR / "aff1ratioofhousepricetoworkplacebasedearnings.xlsx"

print(f"Project root: {PROJECT_ROOT}")
print(f"Raw file path: {RAW_FILE}")


## Download the raw file

The raw workbook is saved in `data/raw/`.

Do not manually edit this file. Any cleaning should be done in code and saved separately in `data/cleaned/`.


In [ ]:
if RAW_FILE.exists():
    print("Raw workbook already exists. Download skipped.")
else:
    print("Downloading ONS workbook...")
    urlretrieve(SOURCE_URL, RAW_FILE)
    print("Download complete.")

file_size_mb = RAW_FILE.stat().st_size / (1024 * 1024)

source_record = pd.DataFrame(
    [{
        "source_name": "ONS House price to workplace-based earnings ratio",
        "source_url": SOURCE_URL,
        "local_file": str(RAW_FILE.relative_to(PROJECT_ROOT)),
        "download_checked_on": date.today().isoformat(),
        "file_size_mb": round(file_size_mb, 2),
    }]
)

source_record


## Inspect workbook sheets

ONS workbooks often include cover sheets, notes and metadata. We inspect the workbook before assuming which sheet contains the actual data table.


In [ ]:
excel_file = pd.ExcelFile(RAW_FILE)

sheet_overview = pd.DataFrame({
    "sheet_number": range(1, len(excel_file.sheet_names) + 1),
    "sheet_name": excel_file.sheet_names,
})

sheet_overview


In [ ]:
for sheet_name in excel_file.sheet_names:
    print("\n" + "=" * 80)
    print(f"Sheet: {sheet_name}")
    print("=" * 80)

    preview = pd.read_excel(
        RAW_FILE,
        sheet_name=sheet_name,
        header=None,
        nrows=12,
    )

    display(preview)


## Identify likely data sheets

This quick scan searches the top of each sheet for words likely to appear in the relevant affordability table. It is a profiling aid, not a substitute for judgement.


In [ ]:
search_terms = [
    "lower quartile",
    "median",
    "earnings",
    "ratio",
    "area",
    "local authority",
]

sheet_matches = []

for sheet_name in excel_file.sheet_names:
    preview = pd.read_excel(
        RAW_FILE,
        sheet_name=sheet_name,
        header=None,
        nrows=30,
        dtype=str,
    )

    text = " ".join(
        preview.fillna("").astype(str).to_numpy().ravel()
    ).lower()

    matched_terms = [term for term in search_terms if term in text]

    sheet_matches.append({
        "sheet_name": sheet_name,
        "matched_terms": ", ".join(matched_terms),
        "match_count": len(matched_terms),
    })

sheet_match_summary = (
    pd.DataFrame(sheet_matches)
    .sort_values(["match_count", "sheet_name"], ascending=[False, True])
)

sheet_match_summary


## Find possible header rows

Some official workbooks have titles or notes above the table. This helper finds rows that look like potential column headers.


In [ ]:
def normalise_text(value):
    if pd.isna(value):
        return ""
    return re.sub(r"\s+", " ", str(value)).strip().lower()


def find_possible_header_rows(sheet_name, max_rows=40):
    preview = pd.read_excel(
        RAW_FILE,
        sheet_name=sheet_name,
        header=None,
        nrows=max_rows,
        dtype=str,
    )

    candidates = []

    for row_number, row in preview.iterrows():
        row_text = " | ".join(
            normalise_text(value) for value in row.tolist() if normalise_text(value)
        )

        score = sum(
            term in row_text
            for term in ["area", "code", "year", "median", "lower", "quartile", "earnings", "ratio"]
        )

        if score >= 3:
            candidates.append({
                "sheet_name": sheet_name,
                "zero_based_row": row_number,
                "excel_row": row_number + 1,
                "score": score,
                "row_text": row_text[:300],
            })

    return candidates


header_candidates = []

for sheet_name in excel_file.sheet_names:
    header_candidates.extend(find_possible_header_rows(sheet_name))

header_candidates_df = pd.DataFrame(header_candidates)

header_candidates_df.sort_values(
    ["score", "sheet_name", "excel_row"],
    ascending=[False, True, True],
).head(20)


## Load the most likely table

After reviewing the previews above, you can override `DATA_SHEET` and `HEADER_ROW` if needed.

The default below selects the highest-scoring candidate. That is useful for a first pass, but the choice should still be checked by eye.


In [ ]:
if not header_candidates_df.empty:
    best_candidate = header_candidates_df.sort_values(
        ["score", "sheet_name", "excel_row"],
        ascending=[False, True, True],
    ).iloc[0]

    DATA_SHEET = best_candidate["sheet_name"]
    HEADER_ROW = int(best_candidate["zero_based_row"])
else:
    DATA_SHEET = excel_file.sheet_names[0]
    HEADER_ROW = 0

print(f"Selected sheet: {DATA_SHEET}")
print(f"Selected header row: {HEADER_ROW} zero-based, Excel row {HEADER_ROW + 1}")

raw_df = pd.read_excel(
    RAW_FILE,
    sheet_name=DATA_SHEET,
    header=HEADER_ROW,
)

print(f"Rows: {raw_df.shape[0]:,}")
print(f"Columns: {raw_df.shape[1]:,}")

raw_df.head()


## Initial data profile

This is the first quality review. The cleaning notebook will make final decisions about field names, filters and data types.


In [ ]:
column_profile = pd.DataFrame({
    "column_name": raw_df.columns.astype(str),
    "dtype": [str(dtype) for dtype in raw_df.dtypes],
    "non_null_count": raw_df.notna().sum().values,
    "missing_count": raw_df.isna().sum().values,
    "missing_pct": (raw_df.isna().mean().values * 100).round(2),
})

column_profile


In [ ]:
column_lookup = pd.DataFrame({
    "column_name": raw_df.columns.astype(str),
    "normalised_column_name": [
        normalise_text(column) for column in raw_df.columns.astype(str)
    ],
})

keywords = ["area", "code", "year", "lower", "quartile", "median", "earnings", "ratio", "price"]

likely_relevant_columns = column_lookup[
    column_lookup["normalised_column_name"].apply(
        lambda name: any(keyword in name for keyword in keywords)
    )
].copy()

likely_relevant_columns


In [ ]:
duplicate_row_count = raw_df.duplicated().sum()
print(f"Exact duplicate rows: {duplicate_row_count:,}")

profile_output = CLEANED_DIR / "initial_column_profile.csv"
source_record_output = RAW_DIR / "source_record.csv"

column_profile.to_csv(profile_output, index=False)
source_record.to_csv(source_record_output, index=False)

print(f"Saved column profile to: {profile_output.relative_to(PROJECT_ROOT)}")
print(f"Saved source record to: {source_record_output.relative_to(PROJECT_ROOT)}")


## Next step

The next notebook will clean and reshape the data.

Before moving on, confirm:
1. the correct data sheet;
2. the area code and area name fields;
3. the year field;
4. the lower-quartile house price, earnings and affordability ratio fields;
5. whether national, regional and local authority rows are mixed together.

Do not remove unusual values without checking them. The most unaffordable areas may look like outliers, but they are likely to be analytically important.
